In [64]:
import os

from io import StringIO
from google.cloud import storage
from dotenv import load_dotenv

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F

from transformers import AutoModelForCausalLM
from transformers import AutoTokenizer, EsmForMaskedLM
from tokenizers import Tokenizer
from peft import get_peft_model, LoraConfig, TaskType


In [65]:
from plm_compare_progen2 import *
from plm_compare_esm import *
from protein_data import *
from pro_gen2_lora import *

In [8]:
# some functions
# inserts mutation amino acid in corresponding position in protein sequence
def insert_wt(seq, pos, wt_aa):
    seq_list = list(seq)
    pos = int(pos)
    if pos < len(seq_list):
        seq_list[pos] = wt_aa
    return ''.join(seq_list)

# computes the ranking loss between two iterables
def listwise_ranking_loss(preds, targets):
    indices = targets.sort(descending=True).indices
    preds = torch.gather(preds, dim=-1, index=indices)
    cumsums = preds.exp().flip(dims=[-1]).cumsum(dim=-1).flip(dims=[-1])
    loss = torch.log(cumsums + 1e-10) - preds
    return loss.mean()

def FineTune_ProGen2_LORA(device, base_model, tokenizer, lora_config, 
                          protein_seq, target_tensor, loss_fn, 
                          lrate=-1e-5, num_epochs=5, k=0.5, 
                          num_samples=20, print_info=True):
    '''
    device is GPU or CPU
    base_model is ProGen2 model
    tokenizer is ProGen2 tokenizer
    lora_config is data for peft lora fine tuning 
    lr is learning rate for gradien descent on new layer
    protein sequence is the sequence lora layer is being trained on
    target_tensor is what loss is measured against initially this is the actual protein sequence
    loss_fn is loss function used in training and validation
    num_epochs is the number of training steps
    k is the proportion of sequence used in training. validation and test indices are created as half the remaining indices each
    num_samples is the number of sample drawn for each epoch used for training and validation
    '''
    amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
    aa_token_ids = [tokenizer.convert_tokens_to_ids(aa) for aa in amino_acids]

    model = get_peft_model(base_model, lora_config)

    model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lrate)

    exp_tensor = target_tensor
    seq_len = exp_tensor.shape[0]

    all_indices = torch.randperm(seq_len)

    num_train = int(seq_len * k)
    num_val   = int(seq_len * k/2)
    num_test  = seq_len - num_train - num_val

    train_indices = all_indices[:num_train] # len 900
    val_indices   = all_indices[num_train:num_train + num_val]
    test_indices  = all_indices[num_train + num_val:]

    train_losses = []
    val_losses = []
    early_stop_count = 0


    vocab_dict = tokenizer.get_vocab()
    seq_list = list(protein_seq)
    inputs = tokenizer(protein_seq, return_tensors="pt").to(device)

    for epoch in range(num_epochs):

        model.train()
        outputs = model(**inputs)
        logits = outputs.logits.squeeze(0)  # (seq_len, vocab_size)
        wt_logits = torch.log_softmax(logits, dim=-1)
        residue_indices = torch.arange(len(protein_seq))

        # ignoring BOS token
        seq_indices = [vocab_dict[aa] for aa in seq_list]
        wt_norm_tensor = wt_logits[residue_indices, seq_indices].unsqueeze(-1)
        LLR_tensor = wt_logits - wt_norm_tensor
        LLR_tensor_aa_only = LLR_tensor[:, aa_token_ids]

        # flatten the LLR_tensor
        flattened_LLR_tensor = LLR_tensor_aa_only.flatten()
        flattened_exp_tensor = exp_tensor.to(device)
        flattened_LLR_tensor = flattened_LLR_tensor.to(device)

        #to stack
        combined = torch.stack([flattened_LLR_tensor, flattened_exp_tensor], dim=0)
        ft_tensor = torch.transpose(combined, 0, 1)

        # predicted_scores = []
        # experimental_values = []

        train_tensor = ft_tensor[train_indices]

        #drop nan
        train_tensor = train_tensor[~torch.any(train_tensor.isnan(), dim=1)]

        # num_samples = num_samples
        positions = train_tensor[torch.randperm(len(train_tensor))[:num_samples]]

        predicts = positions[:, 0] #LLR
        targets = positions[:, 1] #exp

        # Compute loss with predicts and targets
        loss = loss_fn(predicts, targets)
        # log_probs = torch.log_softmax(logits, dim=-1)
        # loss = -log_probs[torch.arange(len(seq_indices)), seq_indices].mean() # needs to be difference of predict/target
        loss.backward()
        # for name, param in model.named_parameters():
        #     if param.requires_grad:
        #         print(name, param.grad.abs().mean())
        optimizer.step()
        optimizer.zero_grad()

        train_losses.append(loss.item())

        # ----------- VALIDATION (no backprop) -----------
        model.eval()
        with torch.no_grad():
            val_tensor = ft_tensor[val_indices]
            val_tensor = val_tensor [~torch.any(val_tensor.isnan(), dim=1)]

            # num_samples = 45
            positions = val_tensor[torch.randperm(len(val_tensor))[:num_samples]]

            predicts = positions[:, 0] #LLR
            targets = positions[:, 1] #exp

            val_loss = loss_fn(predicts, targets)
            # val_loss = listwise_ranking_loss(predicts, targets)
            #   val_log_probs = torch.log_softmax(logits, dim=-1)
            #   val_loss = -log_probs[torch.arange(len(seq_indices)), seq_indices].mean()
            val_losses.append(val_loss.item())

        if print_info==True:
            print(f"Epoch {epoch+1} - Training Loss: {loss.item():.4f} | Validation Loss: {val_loss.item():.4f}")

            if val_loss.item() > loss.item():
                early_stop_count += 1
            else:
                early_stop_count = 0
                
            print(f"Validation loss has exceeded training loss {early_stop_count} time(s) in a row")
            # if early_stop_count > 2:
            #     print("Validation loss exceeded training loss 3 times — early stopping.")
            #     break

    return model, val_loss
# , ft_tensor, test_indices

In [ ]:
# with open('/Users/johnhutchens/Desktop/Practicum/Data/Wild_Dictionaries/pg2_ProGym_matrices.pickle',
#            'rb') as f:
#     pg_dict = pickle.load(f)

In [4]:
# some functions
# inserts mutation amino acid in corresponding position in protein sequence
def insert_wt(seq, pos, wt_aa):
    seq_list = list(seq)
    pos = int(pos)
    if pos < len(seq_list):
        seq_list[pos] = wt_aa
    return ''.join(seq_list)

# computes the ranking loss between two iterables
def listwise_ranking_loss(preds, targets):
    indices = targets.sort(descending=True).indices
    preds = torch.gather(preds, dim=-1, index=indices)
    cumsums = preds.exp().flip(dims=[-1]).cumsum(dim=-1).flip(dims=[-1])
    loss = torch.log(cumsums + 1e-10) - preds
    return loss.mean()

def FineTune_ProGen2_LORA(device, base_model, tokenizer, lora_config, 
                          protein_seq, target_tensor, loss_fn, 
                          lrate=-1e-5, num_epochs=5, k=0.5, 
                          num_samples=20, print_info=True):
    '''
    device is GPU or CPU
    base_model is ProGen2 model
    tokenizer is ProGen2 tokenizer
    lora_config is data for peft lora fine tuning 
    lr is learning rate for gradien descent on new layer
    protein sequence is the sequence lora layer is being trained on
    target_tensor is what loss is measured against initially this is the actual protein sequence
    loss_fn is loss function used in training and validation
    num_epochs is the number of training steps
    k is the proportion of sequence used in training. validation and test indices are created as half the remaining indices each
    num_samples is the number of sample drawn for each epoch used for training and validation
    '''
    amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
    aa_token_ids = [tokenizer.convert_tokens_to_ids(aa) for aa in amino_acids]

    model = get_peft_model(base_model, lora_config)

    model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lrate)

    exp_tensor = target_tensor
    seq_len = exp_tensor.shape[0]

    all_indices = torch.randperm(seq_len)

    num_train = int(seq_len * k)
    num_val   = int(seq_len * k/2)
    num_test  = seq_len - num_train - num_val

    train_indices = all_indices[:num_train] # len 900
    val_indices   = all_indices[num_train:num_train + num_val]
    test_indices  = all_indices[num_train + num_val:]

    train_losses = []
    val_losses = []
    early_stop_count = 0


    vocab_dict = tokenizer.get_vocab()
    seq_list = list(protein_seq)
    inputs = tokenizer(protein_seq, return_tensors="pt").to(device)

    for epoch in range(num_epochs):

        model.train()
        outputs = model(**inputs)
        logits = outputs.logits.squeeze(0)  # (seq_len, vocab_size)
        wt_logits = torch.log_softmax(logits, dim=-1)
        residue_indices = torch.arange(len(protein_seq))

        # ignoring BOS token
        seq_indices = [vocab_dict[aa] for aa in seq_list]
        wt_norm_tensor = wt_logits[residue_indices, seq_indices].unsqueeze(-1)
        LLR_tensor = wt_logits - wt_norm_tensor
        LLR_tensor_aa_only = LLR_tensor[:, aa_token_ids]

        # flatten the LLR_tensor
        flattened_LLR_tensor = LLR_tensor_aa_only.flatten()
        flattened_exp_tensor = exp_tensor.to(device)
        flattened_LLR_tensor = flattened_LLR_tensor.to(device)

        #to stack
        combined = torch.stack([flattened_LLR_tensor, flattened_exp_tensor], dim=0)
        ft_tensor = torch.transpose(combined, 0, 1)

        # predicted_scores = []
        # experimental_values = []

        train_tensor = ft_tensor[train_indices]

        #drop nan
        train_tensor = train_tensor[~torch.any(train_tensor.isnan(), dim=1)]

        # num_samples = num_samples
        positions = train_tensor[torch.randperm(len(train_tensor))[:num_samples]]

        predicts = positions[:, 0] #LLR
        targets = positions[:, 1] #exp

        # Compute loss with predicts and targets
        loss = loss_fn(predicts, targets)
        # log_probs = torch.log_softmax(logits, dim=-1)
        # loss = -log_probs[torch.arange(len(seq_indices)), seq_indices].mean() # needs to be difference of predict/target
        loss.backward()
        # for name, param in model.named_parameters():
        #     if param.requires_grad:
        #         print(name, param.grad.abs().mean())
        optimizer.step()
        optimizer.zero_grad()

        train_losses.append(loss.item())

        # ----------- VALIDATION (no backprop) -----------
        model.eval()
        with torch.no_grad():
            val_tensor = ft_tensor[val_indices]
            val_tensor = val_tensor [~torch.any(val_tensor.isnan(), dim=1)]

            # num_samples = 45
            positions = val_tensor[torch.randperm(len(val_tensor))[:num_samples]]

            predicts = positions[:, 0] #LLR
            targets = positions[:, 1] #exp

            val_loss = loss_fn(predicts, targets)
            # val_loss = listwise_ranking_loss(predicts, targets)
            #   val_log_probs = torch.log_softmax(logits, dim=-1)
            #   val_loss = -log_probs[torch.arange(len(seq_indices)), seq_indices].mean()
            val_losses.append(val_loss.item())

        if print_info==True:
            print(f"Epoch {epoch+1} - Training Loss: {loss.item():.4f} | Validation Loss: {val_loss.item():.4f}")

            if val_loss.item() > loss.item():
                early_stop_count += 1
            else:
                early_stop_count = 0
                
            print(f"Validation loss has exceeded training loss {early_stop_count} time(s) in a row")
            # if early_stop_count > 2:
            #     print("Validation loss exceeded training loss 3 times — early stopping.")
            #     break

    return model, val_loss
# , ft_tensor, test_indices

In [66]:
path = '/Users/johnhutchens/Desktop/Practicum/Data/Domainome/'

with open(path+"dict_PF00030.pkl", "rb") as f:
    dict_PF00030 = pickle.load(f)

In [67]:
dict_PF00030

{'P05813_PF00030_31': {'DMS_mat': array([[-0.10443451,  0.13707808,  0.02807782, ..., -0.03048719,
                  nan,  0.37133969],
         [-0.47260158, -0.33754298, -1.00977863, ..., -0.37827595,
          -0.68666695, -1.01045355],
         [        nan, -0.46455143,         nan, ..., -0.53509969,
          -0.81186809,         nan],
         ...,
         [ 0.01572358,         nan,  0.13154921, ...,  0.00885801,
          -0.10067819,  0.00898411],
         [ 0.12593207, -0.19203179,  0.00742594, ..., -0.22169962,
          -0.51913589, -0.11086738],
         [        nan, -0.23419564, -0.15285446, ..., -0.14229641,
          -0.19328258, -0.29164243]], shape=(89, 20)),
  'wt_seq': 'WKITIYDQENFQGKRMEFTSSCPNVSERSFDNVRSLKVESGAWIGYEHTSFCGQQFILERGEYPRWDAWSGSNAYHIERLMSFRPICSA'},
 'P07315_PF00030_4': {'DMS_mat': array([[        nan,         nan,         nan, ..., -0.03188447,
                  nan,         nan],
         [-0.25448225, -0.05407404,         nan, ...,  0.04906627,
    

In [68]:
for key in keys:
    print(len(dict_PF00030[key]['DMS_mat']))

89
76
80
84
87
81
89
81
84
83
87
78


In [37]:
keys = list(dict_PF00030.keys())

Create testing data from PF00030 to test model trained on P07316_PF00030_87

In [4]:
# load_dotenv()
# cred_path = os.getenv('GOOGLE_APPLICATION_CREDENTIALS')

# os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = cred_path

# client = storage.Client()
# bucket = client.bucket('domainome-data')
# blob = bucket.blob('SupplementaryTable2.txt')

# df = pd.read_csv(StringIO(blob.download_as_text()), sep='\t')

In [ ]:
# df.head()

,domain_ID,uniprot_ID,aa_seq,wt_aa,position,mut_aa,STOP,input_count_rep1,input_count_rep2,input_count_rep3,output_count_rep1,output_count_rep2,output_count_rep3,mean_input_count,fitness,fitness_sigma,normalized_fitness,normalized_fitness_sigma,quality_rank
0,A0A2R8Y422_PF00240_2,A0A2R8Y422,*IFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,Q,2.0,*,True,118.0,113.0,62.0,10.0,29.0,2.0,97.66667,0.030945,0.014885,-0.819050,0.208478,339
1,A0A2R8Y422_PF00240_2,A0A2R8Y422,AIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,Q,2.0,A,False,219.0,277.0,217.0,86.0,225.0,137.0,237.66670,0.069376,0.006673,-0.280790,0.093461,339
2,A0A2R8Y422_PF00240_2,A0A2R8Y422,CIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,Q,2.0,C,False,706.0,726.0,459.0,768.0,507.0,616.0,630.33330,0.082052,0.004141,-0.103250,0.057995,339
3,A0A2R8Y422_PF00240_2,A0A2R8Y422,DIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,Q,2.0,D,False,407.0,431.0,323.0,508.0,159.0,111.0,387.00000,0.071003,0.005162,-0.258003,0.072296,339
4,A0A2R8Y422_PF00240_2,A0A2R8Y422,EIFVKTLMGKTITLEVELSDTIDNVKAKIQDKEGIPPDQQRLIFAG...,Q,2.0,E,False,37.0,56.0,37.0,201.0,102.0,95.0,43.33333,0.116326,0.012085,0.376783,0.169263,339


Train fine tuned model

In [69]:
device = 'cpu'
print(f"Using {device} device")
model_name = "hugohrban/progen2-medium"
base_model, tokenizer = initialize_progen2_noeval(model_name)

Using cpu device


Fine tune model using 30 epochs and learning rate of 1e-3

In [38]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    # target_modules=["query", "key", "value", "output.dense"],
    target_modules=["qkv_proj", "out_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.FEATURE_EXTRACTION
    # task_type=TaskType.CAUSAL_LM
)

# protein_seq = 'EDGINLEEIREFAKNFKIRRLSLGLTQTQVGQALTATEGPAYSQSAICRFEKLDITPKSAQKLKPVLEKWLNEAELRNQEGQQNLMEFVG'
key = keys[0]
protein_seq = dict_PF00030[key]['wt_seq']

df_mutation = pd.read_csv('mutation.csv') # automate df_mutation
fitness_data = df_mutation
fitness_data.reset_index(drop=True, inplace=True)
positions = np.arange(len(protein_seq))

amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
aa_token_ids = [tokenizer.convert_tokens_to_ids(aa) for aa in amino_acids]
positions_col = np.repeat(positions, len(amino_acids))
amino_acids_col = np.tile(list(amino_acids), len(positions))
df2 = pd.DataFrame({'real_position': positions_col, 'mut_aa': amino_acids_col})
df_merged = df2.merge(
    fitness_data,
    on=['real_position', 'mut_aa'],
    how='left')
fitness_list = df_merged['normalized_fitness'].tolist()
fitness_tensor = torch.tensor(fitness_list)
exp_tensor = fitness_tensor
seq_len = exp_tensor.shape[0]

loss = listwise_ranking_loss


# num_epochs = [30]
# l_rates = [1e-3, 5e-3]
# len_ep = len(num_epochs)
# len_lr = len(l_rates)

# results = {}


# for i in range(len_ep):
#     eps = num_epochs[i]
#     results[eps] = []
#     # print(f"Number of epochs: {eps}")
#     for j in range(len_lr):
#         lr = l_rates[j]
#         # print(f"Learning rate: {lr}")
eps = 30
lr = 1e-3
        
model, val_loss = FineTune_ProGen2_LORA(device, base_model, tokenizer, 
                                        lora_config, protein_seq, exp_tensor, 
                                        loss, lrate=lr, num_epochs=eps, k=0.5, 
                                        num_samples=20, print_info=False)

print(f"Validation loss = {val_loss}")
        

Validation loss = 2.1665737628936768


In [ ]:
# save a model
# model_dir = '/Users/johnhutchens/Desktop/Practicum/Models/'+'pg2_lora_eps30_lr1eneg3_PF00030'

# model.save_pretrained(model_dir)
# tokenizer.save_pretrained(model_dir)

('/Users/johnhutchens/Desktop/Practicum/Models/pg2_lora_eps30_lr1eneg3_PF00030/tokenizer_config.json',
 '/Users/johnhutchens/Desktop/Practicum/Models/pg2_lora_eps30_lr1eneg3_PF00030/special_tokens_map.json',
 '/Users/johnhutchens/Desktop/Practicum/Models/pg2_lora_eps30_lr1eneg3_PF00030/vocab.json',
 '/Users/johnhutchens/Desktop/Practicum/Models/pg2_lora_eps30_lr1eneg3_PF00030/merges.txt',
 '/Users/johnhutchens/Desktop/Practicum/Models/pg2_lora_eps30_lr1eneg3_PF00030/added_tokens.json',
 '/Users/johnhutchens/Desktop/Practicum/Models/pg2_lora_eps30_lr1eneg3_PF00030/tokenizer.json')

In [70]:
for key in dict_PF00030.keys():
    seq = dict_PF00030[key]['wt_seq']
    lp, rlp, llr = collect_log_prob_pg2(seq, model, tokenizer)
    dict_PF00030[key]['llr_pg2_lora_eps30_lr1eneg3_PF00030'] = llr

In [71]:
for key in dict_PF00030.keys():
    seq = dict_PF00030[key]['wt_seq']
    lp, rlp, llr = collect_log_prob_pg2(seq, base_model, tokenizer)
    dict_PF00030[key]['llr_pg2'] = llr

In [72]:
keys = list(dict_PF00030.keys())

In [73]:
i = 0

dict_PF00030[keys[i]]['llr_pg2'] - dict_PF00030[keys[i]]['llr_pg2_lora_eps30_lr1eneg3_PF00030']

array([[-2.59404   , -2.0913162 , -3.4876025 , ..., -2.593216  ,
         0.        , -0.77513885],
       [-3.553444  , -3.9058225 , -4.533386  , ..., -3.706253  ,
        -4.620201  , -4.8306885 ],
       [-2.7412338 , -4.2763214 , -5.300476  , ..., -1.0279007 ,
        -5.292282  , -4.0886993 ],
       ...,
       [-2.5057983 ,  0.        , -2.6578522 , ..., -2.8506927 ,
        -6.108902  , -2.8343887 ],
       [-1.4111786 ,  0.2569732 , -3.8256226 , ..., -2.8599472 ,
        -5.49057   , -3.2911375 ],
       [ 0.        , -4.3262253 , -3.6228716 , ..., -3.092659  ,
        -7.2896194 , -5.860054  ]], shape=(89, 20), dtype=float32)

In [ ]:
# path = '/Users/johnhutchens/Desktop/Practicum/Data/Domainome/'
# with open(path+"dict_PF00030.pkl", "wb") as f:
#     pickle.dump(dict_PF00030, f)

In [80]:
for key in keys:
    base_llr = dict_PF00030[key]['llr_pg2']
    lora_llr = dict_PF00030[key]['llr_pg2_lora_eps30_lr1eneg3_PF00030']
    dms_mat = dict_PF00030[key]['DMS_mat']

    # print(len(base_llr), len(lora_llr), len(dms_mat))


    spb = spearman_ignore_nan(base_llr, dms_mat)
    spl = spearman_ignore_nan(lora_llr, dms_mat)

    print(f"Base model spearman correlation with DMS is {spb[0]}")
    print(f"Lora model spearman correlation with DMS is {spl[0]}")
    print()

Base model spearman correlation with DMS is 0.4172571483248994
Lora model spearman correlation with DMS is 0.15932049492101447

Base model spearman correlation with DMS is 0.33897700677632076
Lora model spearman correlation with DMS is 0.17179156530206946

Base model spearman correlation with DMS is 0.36205722660833856
Lora model spearman correlation with DMS is 0.12497598169493365

Base model spearman correlation with DMS is 0.21845045525531706
Lora model spearman correlation with DMS is 0.16575524324784763

Base model spearman correlation with DMS is 0.3662332108928841
Lora model spearman correlation with DMS is 0.05460478145640364

Base model spearman correlation with DMS is 0.2790765656023974
Lora model spearman correlation with DMS is 0.07376780266484208

Base model spearman correlation with DMS is 0.43572135931376255
Lora model spearman correlation with DMS is 0.08960826853333739

Base model spearman correlation with DMS is 0.31101513899232713
Lora model spearman correlation with

In [50]:
dict_PF00030[keys[2]]['DMS_mat'] - dict_PF00030[keys[3]]['DMS_mat']

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(89, 20))